# CarRacing (Gymnasium) × PPO — Colab で All Run

**ランタイム → ランタイムのタイプを変更 → GPU (T4 以上推奨)** にしてから `すべてのセルを実行` してください。
学習は Google Drive にチェックポイント保存され、切断されても再実行で続きから再開します。

In [ ]:
# 1. 依存関係のインストール (Box2D には swig が必要)
!apt-get -qq install -y swig > /dev/null
!pip -q install "gymnasium[box2d]" "stable-baselines3[extra]" imageio imageio-ffmpeg tensorboard

In [ ]:
# 2. ソース取得
import os
if not os.path.exists('/content/CarRacing-for-Colabratory'):
    !git clone https://github.com/mizugamiaqua/CarRacing-for-Colabratory.git /content/CarRacing-for-Colabratory
%cd /content/CarRacing-for-Colabratory

In [ ]:
# 3. Google Drive をマウント (チェックポイント/結果の保存先)
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/carracing'
STEPS = 4_000_000   # 目安: T4 で約 2〜3 時間。動作確認だけなら 200_000

# --- 学習モード切替 -------------------------------------------------
# "scratch" : 最初から学習 (Drive に途中経過があれば自動で再開)
# "finetune": 既存の学習済みモデルの重みから追加学習 → 短時間で完走レベルに到達しやすい
# "skip"    : 学習済みなら学習せず評価だけ行う
MODE = "scratch"
INIT_FROM = f"{OUT}/final_model.zip"   # finetune の元モデル (Drive パス or https:// URL)
if MODE == "finetune":
    STEPS = 1_000_000   # 追加学習のステップ数 (目安: T4 で 30〜60 分)

In [ ]:
# 4. 学習 (再実行すると最新チェックポイントから再開)
if MODE == "skip":
    !python -m carracing.train --out $OUT --skip-if-done --steps $STEPS
elif MODE == "finetune":
    # 追加学習は別フォルダに保存し、元のモデルを上書きしない
    OUT = OUT if OUT.endswith("_ft") else OUT + "_ft"
    !python -m carracing.train --out $OUT --steps $STEPS --init-from "$INIT_FROM"
else:
    !python -m carracing.train --out $OUT --steps $STEPS

In [ ]:
# 5. 学習曲線 (TensorBoard)
%load_ext tensorboard
%tensorboard --logdir $OUT/tb

In [ ]:
# 6. 評価: 10エピソードの平均スコアと周回完走率、動画を保存
import os
model = f'{OUT}/best/best_model.zip' if os.path.exists(f'{OUT}/best/best_model.zip') else f'{OUT}/final_model.zip'
!python -m carracing.evaluate --model $model --episodes 10 --video /content/result.mp4

In [ ]:
# 7. 動画を表示
from IPython.display import Video
Video('/content/result.mp4', embed=True, width=480)